# Callbacks 回调机制

对标文档：[LangChain > Core Components > Callbacks](https://python.langchain.com/docs/concepts/callbacks/)

In [1]:
from langchain_core.callbacks import BaseCallbackHandler, StdOutCallbackHandler
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_learning.llm import get_llm

llm = get_llm()

**术语：**
- **Callback（回调）** = 在事件发生时被调用的函数，用于监听、日志、监控
- **事件** = LLM 开始/结束、链开始/结束、工具调用等生命周期的节点

```mermaid
%%{init: {'theme': 'dark'}}%%
graph LR
    A[LLM 调用] --> B[on_llm_start]
    B --> C[on_llm_new_token 流式]
    C --> D[on_llm_end]
    E[链执行] --> F[on_chain_start]
    F --> G[on_chain_end]
    H[工具调用] --> I[on_tool_start]
    I --> J[on_tool_end]
```

### 1. StdOutCallbackHandler

LangChain 内置的控制台回调处理器，自动打印执行过程中的关键事件。

In [2]:
handler = StdOutCallbackHandler()

chain = ChatPromptTemplate.from_template("解释{topic}（一句话）") | llm | StrOutputParser()
result = chain.invoke({"topic": "回调函数"}, config={"callbacks": [handler]})
print(f"\n结果: {result}")



> Entering new RunnableSequence chain...


> Entering new ChatPromptTemplate chain...

> Finished chain.


> Entering new StrOutputParser chain...

> Finished chain.

> Finished chain.

结果: 回调函数是将一个函数作为参数传递给另一个函数，并在特定事件或条件满足时被调用的函数。


**术语：**
- **StdOutCallbackHandler** = 标准输出回调处理器，把事件信息打印到控制台
- **config={"callbacks": [handler]}** = 通过调用配置传入回调

运行后会看到链的启动/结束、LLM 的启动/结束等事件日志，帮助理解链的执行流程。

### 2. 自定义回调处理器

继承 `BaseCallbackHandler` 实现自定义逻辑，如记录耗时、收集统计等。

In [3]:
import time

class TimerCallback(BaseCallbackHandler):
    """记录 LLM 调用耗时的回调"""
    def on_llm_start(self, serialized, prompts, **kwargs):
        self.start_time = time.time()
        print(f"[开始] LLM 调用...")

    def on_llm_end(self, response, **kwargs):
        elapsed = time.time() - self.start_time
        print(f"[结束] 耗时: {elapsed:.2f}s")

    def on_llm_error(self, error, **kwargs):
        print(f"[错误] {error}")

timer = TimerCallback()
result = llm.invoke("1+1=", config={"callbacks": [timer]})
print(f"结果: {result.content}")

[开始] LLM 调用...
[结束] 耗时: 1.07s
结果: 1+1=2


**术语：**
- **BaseCallbackHandler** = 回调基类，重写感兴趣的事件方法即可
- **on_llm_start** = LLM 开始调用时触发，参数包含 `serialized`（序列化信息）和 `prompts`（提示词）
- **on_llm_end** = LLM 调用成功结束时触发，参数包含 `response`（LLMResult）
- **on_llm_error** = LLM 调用出错时触发

常用回调事件：`on_chat_model_start`、`on_chain_start/end`、`on_tool_start/end/error`、`on_retriever_start/end`。

### 3. 回调的传递方式

回调可以通过多种方式传递到链的各个层级。

In [4]:
class EventCounter(BaseCallbackHandler):
    """统计各事件触发次数"""
    def __init__(self):
        self.events = {}

    def _track(self, name):
        self.events[name] = self.events.get(name, 0) + 1

    def on_llm_start(self, *args, **kwargs): self._track("llm_start")
    def on_llm_end(self, *args, **kwargs): self._track("llm_end")
    def on_chain_start(self, *args, **kwargs): self._track("chain_start")
    def on_chain_end(self, *args, **kwargs): self._track("chain_end")

counter = EventCounter()

# 方式1: 在 invoke 时通过 config 传入
chain = ChatPromptTemplate.from_template("简答{topic}") | llm | StrOutputParser()
chain.invoke({"topic": "Python"}, config={"callbacks": [counter]})

print("事件触发统计:")
for event, count in counter.events.items():
    print(f"  {event}: {count} 次")
print(f"\n总事件数: {sum(counter.events.values())}")

事件触发统计:
  chain_start: 3 次
  chain_end: 3 次
  llm_start: 1 次
  llm_end: 1 次

总事件数: 8


**术语：**
- **config.callbacks** = 调用时传入的回调列表，作用于当前调用及其子调用
- **事件冒泡** = 回调会从子组件（LLM）冒泡到父组件（Chain），所以一个回调可以捕获所有层级的事件

回调传递的三种方式：
1. `invoke(..., config={"callbacks": [handler]})` — 调用时传入（推荐）
2. `chain.with_config(callbacks=[handler])` — 绑定到组件的配置上
3. 在链构造时通过组件参数传入（不推荐，易遗漏）